In [1]:
#from sys import path as syspath
#from os import path as ospath

#syspath.insert(1, r'D:\ARTM\topic-modelling-attention\src')
#syspath

['C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none\\python313.zip',
 'D:\\ARTM\\topic-modelling-attention\\src',
 'C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none\\DLLs',
 'C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none\\Lib',
 'C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\builds-v0\\.tmpGHKOtm',
 '',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\builds-v0\\.tmpGHKOtm\\Lib\\site-packages',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages\\win32',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages\\win32\\lib',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages\\Pythonwin',
 '

In [2]:
import jax
import jax.numpy as jnp
import numpy as np 

from sklearn.datasets import fetch_20newsgroups

import matplotlib.pyplot as plt
import seaborn as sns

#from cartm.model import ContextTopicModel
from cartm.preprocessing import DatasetPreprocessor
from cartm.attentive_model import MatveyAttentiveTopicModel


This example will guide you through the basic interaction with the model.

## Подготовка данных

In [3]:
'''data = [
    'If I were you, I would try something new today.',
    'The best time for a new beginning is now.',
    'Small steps every day lead to big results.',
    'Sometimes you need to disconnect to reconnect.'
]'''

with open('../data/test_data.txt') as f:
    data = f.readlines()

print(f'Total number of documents in corpus: {len(data)}')
print(f'Total number of words in corpus: {sum([len(doc.split(" ")) for doc in data])}')

Total number of documents in corpus: 4
Total number of words in corpus: 34


In [4]:
preprocessor = DatasetPreprocessor()
tokenized_data, document_bounds = preprocessor.fit_transform(data)
print(f'Total number of document boundaries in preprocessed corpus: {len(document_bounds)}')
print(f'Total number of tokenized words in preprocessed corpus: {len(tokenized_data)}')

Total number of document boundaries in preprocessed corpus: 5
Total number of tokenized words in preprocessed corpus: 20


In [16]:
vocabulary = preprocessor.vocabulary
print(document_bounds)
#sum(len(doc) for doc in tokenized_data)
print(tokenized_data)
print(preprocessor.vocabulary)
ctx_bounds = document_bounds

[ 0  5  9 16 20]
[18 17 12  8 16  1 15  8  0 11 14  5  3  6  2 10 13  7  4  9]
{'begin': 0, 'best': 1, 'big': 2, 'day': 3, 'disconnect': 4, 'everi': 5, 'lead': 6, 'need': 7, 'new': 8, 'reconnect': 9, 'result': 10, 'small': 11, 'someth': 12, 'sometim': 13, 'step': 14, 'time': 15, 'today': 16, 'tri': 17, 'would': 18}


# Имитация итерации EM-алгоритма

In [10]:
_eps = 1e-12
def _norm(x: jax.Array) -> jax.Array:
    # Нормализация идет по столбцам. Перед этим отрицательные значения
    # обнуляются, чтобы после сложения статистик и градиентов
    # регуляризации не появлялись некорректные вероятности.
    x = jnp.maximum(x, jnp.zeros_like(x))
    norm = x.sum(axis=0)
    return jnp.where(norm > _eps, x / norm, jnp.zeros_like(x))

def _norm_rows(x: jax.Array) -> jax.Array:
    x = jnp.maximum(x, jnp.zeros_like(x))
    norm = x.sum(axis=1, keepdims=True)
    return jnp.where(norm > _eps, x / norm, jnp.zeros_like(x))

# Исходные параметры и переменные

In [18]:
ctx_len = 3 # полуширина окна контекста
n_topics = 10 # количество топиков T 
beta: float = 0.5
gamma_i: float = 0.6 #// коэффициент затухания будущих токенов
gamma_n: float = 0.6 #// коэффициент затухания прошедших токенов
explicit_include_self: bool = True
eps: float = 1e-12

attn_bounds = document_bounds # границы документов - для обрезки окон контекста
print(f"{attn_bounds=}")
vocab_size = len(vocabulary) # размер словаря W
print(f"{vocab_size=}")


attn_bounds=Array([ 0,  5,  9, 16, 20], dtype=int32)
vocab_size=19


In [11]:
# матрица Фи размерность (W,T)
phi = jax.random.uniform(  # инициализация равномерным распределением 
            key=jax.random.key(42),
            shape=(vocab_size, n_topics),
        )
phi = _norm(phi) # нормировка
print(f"{phi.shape=}")

phi.shape=(19, 10)


In [9]:
# вектор-псевдоматрица размерностью (T, ) для хранения распределения топиков (частотности)
# сколько в корпусе (батче) токенов w приходится на топик t
n_t = jnp.full(           
            shape=(n_topics, ),
            fill_value=len(tokenized_data) / n_topics,    # инициализация средним значением
        )  # (T, )
print(f"{n_t=}")

n_t=Array([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.], dtype=float32, weak_type=True)


## Начало итерации (далее - шаг)

В результате шага необходимо получить пересчитанные значения для:
- phi_it - матрица вероятности контекста в позиции i при наличии в нем топика t 
- phi_new - матрица Фи 
- theta - матрица Тета 
- n_t - вектор частот топиков

Для случая пакетной обработки после каждого пакета корректируем phi_new и n_t_new с затуханием (lr = 0.1)

phi_new = phi_new * (1 - lr) + phi_step * lr # значение Фи, полученное после обработки пакета

n_t_new = n_t_new * (1 - lr) + n_t_step * lr # значение вектора частот топиков, полученное после обработки пакета


In [12]:
batch = tokenized_data
p_ti = phi[batch]
x = p_ti.T
x.shape

(10, 20)

In [19]:
def _ema_windowed_attn(
        *,
        x: jax.Array,
        ctx_bounds: jax.Array,
) -> jax.Array:
    if x.shape[1] == 0: #// не обрабатываются векторы?
        return x

    seq_len = x.shape[1] #// для p_ti.T это I
    doc_ids = jnp.searchsorted(ctx_bounds[1:], jnp.arange(seq_len), side='right') #// массив номеров документов для каждой позиции
    doc_starts = jnp.concatenate([ #// позиции начала документа
        jnp.array([True], dtype=bool),
        doc_ids[1:] != doc_ids[:-1],
    ])
    doc_ends = jnp.concatenate([ #// позиции начала документа
        doc_ids[:-1] != doc_ids[1:],
        jnp.array([True], dtype=bool),
    ])

    forward = jnp.zeros_like(x)
    backward = jnp.zeros_like(x)

    # Ограничиваем влияние соседей окном длины ctx_len с каждой стороны.
    # При достаточном окне формула совпадает с EMA внутри документа.
    max_offset = ctx_len if ctx_len > 0 else seq_len - 1

    for offset in range(max_offset + 1):
        if seq_len <= offset:
            break

        decay_forward = (1.0 - gamma_i) ** offset
        decay_backward = (1.0 - gamma_n) ** offset

        if offset == 0:
            same_doc = jnp.ones((seq_len,), dtype=x.dtype)
            src_forward = x
            src_backward = x
            coeff_forward = decay_forward * (
                    gamma_i + (1.0 - gamma_i) * doc_starts.astype(x.dtype)
            )
            coeff_backward = decay_backward * (
                    gamma_n + (1.0 - gamma_n) * doc_ends.astype(x.dtype)
            )
            forward = forward + src_forward * coeff_forward[None, :]
            backward = backward + src_backward * coeff_backward[None, :]
            continue

        same_doc = (doc_ids[offset:] == doc_ids[:-offset]).astype(x.dtype)

        src_forward = x[:, :-offset]
        coeff_forward = decay_forward * (
                gamma_i + (1.0 - gamma_i) * doc_starts[:-offset].astype(x.dtype)
        )
        forward = forward.at[:, offset:].add(
            src_forward * (coeff_forward * same_doc)[None, :]
        )

        src_backward = x[:, offset:]
        coeff_backward = decay_backward * (
                gamma_n + (1.0 - gamma_n) * doc_ends[offset:].astype(x.dtype)
        )
        backward = backward.at[:, :-offset].add(
            src_backward * (coeff_backward * same_doc)[None, :]
        )

    return beta * forward + (1.0 - beta) * backward

In [ ]:
coeff = [i for i in range(ctx_len)]

for i, p in enumerate(x[0]):
    print(i, p)

In [49]:
c = 4
w = 2/(c + 1)
gamma = 1 - 2/(c + 1)

print(f"{gamma=}")
for i in range(10):
    w *= gamma
    print(w)

print([2/(c + 1)*(1 - 2/(c + 1))**i for i in range(0, 6)])
print([(1 - 2/(c + 1))*(2/(c + 1))**i for i in range(0, 6)])

gamma=0.6
0.24
0.144
0.08639999999999999
0.05183999999999999
0.031103999999999993
0.018662399999999996
0.011197439999999998
0.006718463999999999
0.004031078399999999
0.0024186470399999993
[0.4, 0.24, 0.144, 0.08639999999999999, 0.05184, 0.031103999999999993]
[0.6, 0.24, 0.09600000000000002, 0.03840000000000001, 0.015360000000000002, 0.006144000000000001]


In [20]:
_ema_windowed_attn(x=x, ctx_bounds=ctx_bounds)

Array([[0.06923875, 0.07571061, 0.046726  , 0.04001643, 0.01998563,
        0.03970526, 0.0455647 , 0.04513705, 0.05083704, 0.06703639,
        0.04977947, 0.05880874, 0.06637294, 0.0329864 , 0.04465728,
        0.05336193, 0.05568204, 0.05256825, 0.05380183, 0.07572302],
       [0.05817763, 0.03971302, 0.04823151, 0.07262878, 0.05775087,
        0.07924056, 0.0566333 , 0.07799661, 0.06831951, 0.07135339,
        0.0687844 , 0.04647684, 0.04339854, 0.0288983 , 0.02784308,
        0.06498164, 0.07706507, 0.03730053, 0.04922497, 0.02530576],
       [0.0512756 , 0.03540115, 0.04983447, 0.07240903, 0.05238096,
        0.09607154, 0.09160183, 0.08635242, 0.07068764, 0.03953739,
        0.05270606, 0.04490489, 0.04134909, 0.0653968 , 0.02375783,
        0.0405637 , 0.02590958, 0.05664597, 0.04364994, 0.03419797],
       [0.02232295, 0.06073714, 0.06129479, 0.06971322, 0.0651349 ,
        0.07164128, 0.05051007, 0.06609127, 0.05376033, 0.0751683 ,
        0.06978233, 0.07380615, 0.06277827, 0

In [10]:
# phi_hatch - матрица для перевзвешивания Фи с учетом частотности топиков (далее Фи^)
# phi.T умножается element-wise на n_t (частотность топиков), нормируется и транспонируется обратно
# (T, W) * (T, 1) = (T, W) -> t -> (W, T)
print(f"{phi.T.shape=}")
print(f"{n_t[:, None].shape=}")
phi_hatch = _norm(phi.T * n_t[:, None]).T # (W, T)
print(f"{phi_hatch.shape=}")

# в примере 19 строк для каждого токена с частотой соответствующего топика в 10 колонках
# токен 8 - "new", в векторе 10 значений частоты соответствующего топика
print(f"{phi_hatch[8]=}")

phi.T.shape=(10, 19)
n_t[:, None].shape=(10, 1)
phi_hatch.shape=(19, 10)
phi_hatch[8]=array([0.00623954, 0.00542419, 0.04397687, 0.14882576, 0.19154426,
       0.0731359 , 0.13844424, 0.06082839, 0.10824747, 0.22333333],
      dtype=float32)


### Далее расчет Теты с учетом перевзвешенной Фи^ и контекстов

Реализовано в функции 
```
_calc_theta(
    phi_hatch=phi_hatch,   # Фи^
    batch=batch,           # батч (массив токенов, обрабатываемых документов)
    ctx_bounds=ctx_bounds, # границы для обрезки окон контекстов (фактически границы документов)
    )
```

In [11]:
batch = tokenized_data
print(f"{batch=}")
# выделенная часть Фи^, относящаяся к батчу - Фи^_батч
phi_it_hatch = jnp.take_along_axis( 
            phi_hatch,
            indices=batch[:, None],         # берем из Фи^ строки токенов, встречающихся в батче 
            axis=0,
            )
print(f"{phi_it_hatch.shape=}") # размерность матрицы Фи^_батч (позиции в батче Х топики) (C, T)
# phi_it_hatch - матрица 20 строк для каждой позиции (не токена) по порядку в батче (tokenized_data) с 10 колонками вероятности топиков
# пример: токен 8 - "new" стоит в 3 и 7 позиции в батче (tokenized_data)
print(f"{phi_it_hatch[3]=}")
print(f"{phi_it_hatch[7]=}")

batch=Array([18, 17, 12,  8, 16,  1, 15,  8,  0, 11, 14,  5,  3,  6,  2, 10, 13,
        7,  4,  9], dtype=int32)
phi_it_hatch.shape=(20, 10)
phi_it_hatch[3]=Array([0.00623954, 0.00542419, 0.04397687, 0.14882576, 0.19154426,
       0.0731359 , 0.13844424, 0.06082839, 0.10824747, 0.22333333],      dtype=float32)
phi_it_hatch[7]=Array([0.00623954, 0.00542419, 0.04397687, 0.14882576, 0.19154426,
       0.0731359 , 0.13844424, 0.06082839, 0.10824747, 0.22333333],      dtype=float32)


### Трехмерный тензор окон контекста 

Формирование для батча окон контекста для каждой позиции батча и топика

In [12]:
def _get_context_tensor(batch):
    batch_size = batch.shape[0]     # количество позиций в батче B
    pad_token = -1  # assuming we don't have negative tokens in vocabulary
    
    # shifts for rolling the batch along new dimension
    #  [0, -1, -2, ..., -2 * ctx_len - 1]
    shifts = jnp.arange(0, -2 * ctx_len - 1, -1)  # (2C + 1, )
    print(f"{shifts=}")
    
    padding = jnp.full(
        (ctx_len, n_topics),
        fill_value=pad_token,
        dtype=batch.dtype,
    )  # (C, T)
    print(f"{padding.shape=}")
    print(f"{padding=}")
    
    padded_batch = jnp.concatenate([padding, batch, padding], axis=0)
    print(f"{padded_batch.shape=}")
    print(f"{padded_batch=}")
    
    # rolling and clipping each "slice" of batch
    def shift_batch(shift):
        return jnp.roll(padded_batch, shift, axis=0)[:batch_size]
    
    # apply vmap over all shifts
    # размножение батча с учетом контекстов - длина батча Х величина окна Х число топиков
    stacked_tensor = jax.vmap(shift_batch)(shifts).transpose(1, 0, 2)
    print(f"{stacked_tensor.shape=}")
    return stacked_tensor

# phi_it_hatch_with_context - Фи^_батч_контекст: трехмерный массив с измерениями 
#  {длина батча Х величина окна Х число топиков} (B x (2C+1) x T)
# где в ячейке вероятность, которую в заданной позиции i вносит токен из позиции j контекста Ci, для топика t
# окно контекста с учетом границ документов (зануляются токены вне границ)
# в примере получается (20 х 7 х 10)
phi_it_hatch_with_context = _get_context_tensor(batch=phi_it_hatch)
print(f"{phi_it_hatch_with_context.shape=}")

shifts=Array([ 0, -1, -2, -3, -4, -5, -6], dtype=int32)
padding.shape=(3, 10)
padding=Array([[-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.]], dtype=float32)
padded_batch.shape=(26, 10)
padded_batch=Array([[-1.        , -1.        , -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ],
       [-1.        , -1.        , -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ],
       [-1.        , -1.        , -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ],
       [ 0.09118721,  0.11585489,  0.17614725,  0.15402782,  0.12468565,
         0.11698034,  0.04652762,  0.05900681,  0.08696249,  0.02861993],
       [ 0.1055353 ,  0.03661064,  0.05360207,  0.12943278,  0.15812047,
         0.14386164

### Формирование матрицы внимания

In [13]:
batch_size = len(tokenized_data)
# матрица размерностью (len(data) + 2C, 2C + 1)
attn_matrix = jnp.ones(
    shape=(batch_size + ctx_len * 2, ctx_len * 2 + 1),
    dtype=bool,
)  # 
# в примере (20 + 2 * 3, 2 * 3 )
print(attn_matrix.shape) 

(26, 7)


In [14]:
# формирование префиксных масок
prefix_bounds = document_bounds[: -1] + ctx_len
print(prefix_bounds)

ignored_mask_prefix = jnp.ones((ctx_len, ctx_len), dtype=bool)  # (C, C)
ignored_mask_prefix = jnp.rot90(~jnp.triu(ignored_mask_prefix))
print(ignored_mask_prefix)
ignored_mask_prefix = jnp.tile(ignored_mask_prefix, reps=len(prefix_bounds),).T
ignored_mask_prefix

[ 3  8 12 19]
[[False False False]
 [False False  True]
 [False  True  True]]


Array([[False, False, False],
       [False, False,  True],
       [False,  True,  True],
       [False, False, False],
       [False, False,  True],
       [False,  True,  True],
       [False, False, False],
       [False, False,  True],
       [False,  True,  True],
       [False, False, False],
       [False, False,  True],
       [False,  True,  True]], dtype=bool)

In [15]:
shifts = jnp.ones((len(prefix_bounds), ctx_len), dtype=int)  # (B, C)
print(shifts)
shifts = shifts.at[:, 0].set(prefix_bounds)
print(shifts)
shifts = jnp.cumsum(shifts, axis=1)
print(shifts)
shifts = shifts.reshape(-1, 1)  # (B * C, 1)
print(shifts.T)

[[1 1 1]
 [1 1 1]
 [1 1 1]
 [1 1 1]]
[[ 3  1  1]
 [ 8  1  1]
 [12  1  1]
 [19  1  1]]
[[ 3  4  5]
 [ 8  9 10]
 [12 13 14]
 [19 20 21]]
[[ 3  4  5  8  9 10 12 13 14 19 20 21]]


In [16]:
prefix_columns = jnp.arange(ctx_len)  # (C, )
print(prefix_columns)
attn_matrix = attn_matrix.at[shifts, prefix_columns].set(ignored_mask_prefix)
print(attn_matrix.shape)

[0 1 2]
(26, 7)


In [17]:
# формирование суффиксных масок
suffix_bounds = attn_bounds[1:]  # (B, )
ignored_mask_suffix = jnp.ones((ctx_len, ctx_len), dtype=bool)  # (C, C)
print(ignored_mask_suffix)
ignored_mask_suffix = jnp.rot90(~jnp.tril(ignored_mask_suffix))  # (C, C)
# for broadcasting
print(ignored_mask_suffix)
ignored_mask_suffix = jnp.tile(
    ignored_mask_suffix,
    reps=len(suffix_bounds),
).T  # (B * C, C)
print(ignored_mask_suffix)

[[ True  True  True]
 [ True  True  True]
 [ True  True  True]]
[[ True  True False]
 [ True False False]
 [False False False]]
[[ True  True False]
 [ True False False]
 [False False False]
 [ True  True False]
 [ True False False]
 [False False False]
 [ True  True False]
 [ True False False]
 [False False False]
 [ True  True False]
 [ True False False]
 [False False False]]


In [18]:
shifts = jnp.ones((len(suffix_bounds), ctx_len), dtype=int)  # (I, C)
print(shifts)
shifts = shifts.at[:, 0].set(suffix_bounds)
print(shifts)
shifts = jnp.cumsum(shifts, axis=1)
print(shifts)
shifts = shifts.reshape(-1, 1)  # (B * C, 1)
print(shifts.T)

[[1 1 1]
 [1 1 1]
 [1 1 1]
 [1 1 1]]
[[ 5  1  1]
 [ 9  1  1]
 [16  1  1]
 [20  1  1]]
[[ 5  6  7]
 [ 9 10 11]
 [16 17 18]
 [20 21 22]]
[[ 5  6  7  9 10 11 16 17 18 20 21 22]]


In [19]:
suffix_columns = jnp.arange(ctx_len + 1, ctx_len * 2 + 1)  # (C, )
suffix_columns

Array([4, 5, 6], dtype=int32)

In [20]:
ignored_mask_suffix[::-1]

Array([[False, False, False],
       [ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [ True, False, False],
       [ True,  True, False]], dtype=bool)

In [21]:
attn_matrix = attn_matrix.at[shifts[::-1], suffix_columns].set(ignored_mask_suffix[::-1])
attn_matrix

Array([[ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True, False],
       [False,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True, False],
       [ True,

In [22]:
# remove padding
attn_matrix = attn_matrix[ctx_len: -ctx_len]  # (I, 2C + 1)
attn_matrix

Array([[False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True, False],
       [False,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False,

In [23]:
print(document_bounds)
list(enumerate(list(attn_matrix)))

[ 0  5  9 16 20]


[(0, Array([False, False, False,  True,  True,  True,  True], dtype=bool)),
 (1, Array([False, False,  True,  True,  True,  True,  True], dtype=bool)),
 (2, Array([False,  True,  True,  True,  True,  True, False], dtype=bool)),
 (3, Array([ True,  True,  True,  True,  True, False, False], dtype=bool)),
 (4, Array([ True,  True,  True,  True, False, False, False], dtype=bool)),
 (5, Array([False, False, False,  True,  True,  True,  True], dtype=bool)),
 (6, Array([False, False,  True,  True,  True,  True, False], dtype=bool)),
 (7, Array([False,  True,  True,  True,  True, False, False], dtype=bool)),
 (8, Array([ True,  True,  True,  True, False, False, False], dtype=bool)),
 (9, Array([False, False, False,  True,  True,  True,  True], dtype=bool)),
 (10, Array([False, False,  True,  True,  True,  True,  True], dtype=bool)),
 (11, Array([False,  True,  True,  True,  True,  True,  True], dtype=bool)),
 (12, Array([ True,  True,  True,  True,  True,  True,  True], dtype=bool)),
 (13, Arr

In [24]:
def _get_context_weights_1d(gamma: float) -> np.ndarray:
    # значение из класса по умолчанию
    _self_aware_context = False
    # w_i = gamma * (1 - gamma)**i
    # правая половина весов контекста размером C
    suffix_context_weights = np.cumprod(np.full(ctx_len, (1 - gamma))) * gamma  # (C, ) 
    # заполняем ndarray длины self.ctx_len значениями (1 - gamma)
    # кумулятивное произведение [1, 2, 3, 4, 5] -> [1, 2, 6, 24, 120]
    #jax.debug.print("{suffix_context_weights}", suffix_context_weights=suffix_context_weights)
    prefix_context_weights = suffix_context_weights[::-1]  # (C, ) левая половина - перевернутая правая
    self_context_weight = np.array([gamma * _self_aware_context]) # массив из одного элемента _gamma или 0
    context_weights = np.concatenate([
        prefix_context_weights,
        self_context_weight,
        suffix_context_weights,
    ])
    # собранный массив типа ctx_len = 3, gamma = 0.6 -> [0.0384, 0.096 , 0.24  , 0.    , 0.24  , 0.096 , 0.0384]
    return context_weights  # (2C + 1, ) 

# при ctx_len = 3, gamma = 0.6
_context_weights_1d = _get_context_weights_1d(0.6)
print(f"{_context_weights_1d=}")
context_matrix = _context_weights_1d * attn_matrix  # (I, 2C + 1)
list(enumerate(list(context_matrix)))

_context_weights_1d=array([0.0384, 0.096 , 0.24  , 0.    , 0.24  , 0.096 , 0.0384])


[(0,
  Array([0.    , 0.    , 0.    , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (1,
  Array([0.    , 0.    , 0.24  , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (2, Array([0.   , 0.096, 0.24 , 0.   , 0.24 , 0.096, 0.   ], dtype=float32)),
 (3,
  Array([0.0384, 0.096 , 0.24  , 0.    , 0.24  , 0.    , 0.    ], dtype=float32)),
 (4,
  Array([0.0384, 0.096 , 0.24  , 0.    , 0.    , 0.    , 0.    ], dtype=float32)),
 (5,
  Array([0.    , 0.    , 0.    , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (6, Array([0.   , 0.   , 0.24 , 0.   , 0.24 , 0.096, 0.   ], dtype=float32)),
 (7, Array([0.   , 0.096, 0.24 , 0.   , 0.24 , 0.   , 0.   ], dtype=float32)),
 (8,
  Array([0.0384, 0.096 , 0.24  , 0.    , 0.    , 0.    , 0.    ], dtype=float32)),
 (9,
  Array([0.    , 0.    , 0.    , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (10,
  Array([0.    , 0.    , 0.24  , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (11,
  Array([0.    , 0.096 , 0.24  , 0.    , 0.24  , 0.0

In [25]:
# нормирование матрицы весов контекста
# количество строк - позиции в батче 
# в каждой строке нормированные веса с учетом границ документов
context_matrix = _norm(context_matrix.T).T
print(f"{context_matrix=}") 

context_matrix=array([[0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.390625  , 0.        , 0.390625  ,
        0.15625001, 0.06250001],
       [0.        , 0.14285715, 0.35714287, 0.        , 0.35714287,
        0.14285715, 0.        ],
       [0.0625    , 0.15625   , 0.39062497, 0.        , 0.39062497,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.4166667 , 0.        , 0.4166667 ,
        0.16666667, 0.        ],
       [0.        , 0.16666667, 0.4166667 , 0.        , 0.4166667 ,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        

In [26]:
print(f"{context_matrix[..., None].shape=}")
print(f"{phi_it_hatch_with_context.shape=}")

context_matrix[..., None].shape=(20, 7, 1)
phi_it_hatch_with_context.shape=(20, 7, 10)


In [27]:
# element-wise умножение матрицы весов контекста для каждой позиции (20, 7) на
# phi_it_hatch_with_context - трехмерный массив с измерениями длина батча Х величина окна Х число топиков (I Х 2C+1 Х T) (20, 7, 10)
# где в ячейке вероятность, которую в заданной позиции i вносит токен из позиции j в контексте Ci, для топика t
# окно контекста с учетом границ документов (зануляются токены вне границ)
# после умножения вероятности, взвешенные по весам в контексте
theta_it = context_matrix[..., None] * phi_it_hatch_with_context
print(f"{theta_it.shape=}")

theta_it.shape=(20, 7, 10)


In [28]:
# вот как умнножаются, чтобы получилась theta_it
print(f"{context_matrix[..., None][0][4]=}")
print(f"{phi_it_hatch_with_context[0][4]=}")
print(f"{theta_it[0][4]=}")
print(f"{context_matrix[..., None][0][4] * phi_it_hatch_with_context[0][4]=}")

context_matrix[..., None][0][4]=array([0.64102566], dtype=float32)
phi_it_hatch_with_context[0][4]=Array([0.1055353 , 0.03661064, 0.05360207, 0.12943278, 0.15812047,
       0.14386164, 0.10507683, 0.11007395, 0.08883123, 0.06885507],      dtype=float32)
theta_it[0][4]=Array([0.06765083, 0.02346836, 0.0343603 , 0.08296973, 0.10135928,
       0.092219  , 0.06735694, 0.07056022, 0.0569431 , 0.04413787],      dtype=float32)
context_matrix[..., None][0][4] * phi_it_hatch_with_context[0][4]=Array([0.06765083, 0.02346836, 0.0343603 , 0.08296973, 0.10135928,
       0.092219  , 0.06735694, 0.07056022, 0.0569431 , 0.04413787],      dtype=float32)


In [29]:
# суммирование по измерению окна контекста - остаются позиции в батче и топики
# в результате вероятность топика в данной позиции, полученная по токенам в окне контекста 
theta_it = jnp.sum(theta_it, axis=1)  # (I, T)
print(f"{theta_it.shape=}")

theta_it.shape=(20, 10)


## Расчет матрицы распределения вероятности топика для контекстов в позициях p_ti

In [30]:
def _calc_p_ti(
        *,
        phi: jax.Array,
        theta: jax.Array,
        batch: jax.Array
) -> tuple[jax.Array, jax.Array]:
    # выделенная часть Фи, относящаяся к батчу - Фи_батч
    phi_it = jnp.take_along_axis(
        phi,
        indices=batch[:, None],         # берем из Фи строки токенов, встречающихся в батче 
        axis=0,
    )  # (I, T)
    print(f"{phi_it.shape=}")
    # element-wise умножение Фи (в части токенов, относящихся к батчу) на Тету
    # обе матрицы размерностью ((I, T))
    # нормализация по строкам - вероятность топиков для каждой позиции в сумме единица
    print(f"{theta.shape=}")
    print(f"{(phi_it * theta).shape=}")
    p_ti = _norm((phi_it * theta).T).T  # (I, T)
    print(f"{p_ti.shape=}")
    print()
    print("# element-wise")
    print(f"{phi_it[0]=}")
    print(f"{theta[0]=}")
    print(f"{(phi_it * theta)[0]=}")
    
    print(f"{phi_it[0][1]=}")
    print(f"{theta[0][1]=}")
    print(f"{(phi_it * theta)[0][1]=}")
    print(f"{phi_it[0][1] * theta[0][1]=}")
    
    return p_ti, phi_it

# phi_it - Фи (в части токенов, относящихся к батчу) вероятность темы при наличии токена w в позиции i
# theta_it - Тета вероятность топика в данной позиции, полученная по токенам в окне контекста позиции i
# p_ti - вероятность топика в этой позиции при наличии этого токена и окружающего его контекста
p_ti, phi_it = _calc_p_ti(
    phi=phi,
    theta=theta_it,
    batch=batch,
)  # (I, T)


phi_it.shape=(20, 10)
theta.shape=(20, 10)
(phi_it * theta).shape=(20, 10)
p_ti.shape=(20, 10)

# element-wise
phi_it[0]=Array([0.04269939, 0.05425029, 0.08248284, 0.07212517, 0.05838539,
       0.05477729, 0.02178705, 0.02763057, 0.04072112, 0.01340159],      dtype=float32)
theta[0]=Array([0.10762397, 0.04735994, 0.05892551, 0.10403094, 0.15768561,
       0.12226825, 0.11663017, 0.11516932, 0.09098475, 0.07932156],      dtype=float32)
(phi_it * theta)[0]=Array([0.00459548, 0.00256929, 0.00486034, 0.00750325, 0.00920654,
       0.00669752, 0.00254103, 0.00318219, 0.003705  , 0.00106303],      dtype=float32)
phi_it[0][1]=Array(0.05425029, dtype=float32)
theta[0][1]=Array(0.04735994, dtype=float32)
(phi_it * theta)[0][1]=Array(0.00256929, dtype=float32)
phi_it[0][1] * theta[0][1]=Array(0.00256929, dtype=float32)


### Оценка частоты топиков n_t на шаге

In [31]:
# суммируем вероятности топика по всем позициям, получаем оценку частот топиков в батче
def _calc_n_t(*, p_ti):
    return jnp.sum(p_ti, axis=0)  # (T, )

n_t_new = _calc_n_t(p_ti=p_ti)
print(f"{n_t_new=}")

n_t_new=Array([1.7109293, 1.5929288, 1.7314843, 1.9040339, 2.2407615, 2.2293854,
       2.1159027, 1.8696146, 2.1715155, 2.4334447], dtype=float32)


#### Подготовка регуляризатора для пересчета Фи

In [32]:
# обработка регуляризаторов на примере DecorrelationRegularization
def decorrelation_reg(phi_wt: jax.Array) -> float:
    corr_matrix = phi_wt.T @ phi_wt  # (T, T)
    # remove duplicates and diagonal terms
    corr_triu = jnp.triu(corr_matrix, k=1)
    return jnp.sum(corr_triu)

def _compose_regularizations():
    regs = [decorrelation_reg]
    reg_grad = jax.grad(lambda x: sum([1.0, ] + [reg(x) for reg in regs]))
    return jax.jit(reg_grad)

grad_reg = _compose_regularizations()
print(f"{grad_reg=}")

grad_reg=<PjitFunction of <function _compose_regularizations.<locals>.<lambda> at 0x0000020F7AC8C180>>


### Пересчет Фи на шаге

In [33]:
# расчет нового значения Фи по результатам шага
def _calc_phi(
        *,
        batch: jax.Array,
        phi: jax.Array,
        p_ti: jax.Array,
        grad_reg,
    ):
    # jnp.add.at - jax.numpy.ufunc.at(a, indices[, b, inplace])
    # применяет функцию ufunc к элементам a по индексам indices, b - аргумент, inplace=True имитирует по_месту 
    # здесь создает новый массив на основе нулевого массива формы phi и для токенов, полученных в батче, прибавляет p_ti
    # прибавляет столько раз, сколько встречается в батче
    # в результате матрица размерностью как Фи, но ненормализованная - суммы вероятностей
    # дальше применяем регуляризаторы и нормализуем
    phi_new = jnp.add.at(
        jnp.zeros_like(phi),
        batch,
        p_ti,
        inplace=False,
    )  # (W, T)
    print(f"{phi_new[0]=}")
    phi_new -= phi * grad_reg(phi)  # (W, T)
    print(f"{phi_new[0]=}")
    phi_new = _norm(phi_new)  # (W, T)
    print(f"{phi_new[0]=}")
    return phi_new

phi_new = _calc_phi( 
            batch=batch,
            phi=phi,
            p_ti=p_ti,
            grad_reg=grad_reg)


phi_new[0]=Array([0.01548281, 0.07350703, 0.09873346, 0.14708544, 0.0881432 ,
       0.02225126, 0.16040105, 0.03219692, 0.06452698, 0.29767174],      dtype=float32)
phi_new[0]=Array([0.00068657, 0.03804044, 0.07118985, 0.11466126, 0.072062  ,
       0.01310876, 0.1280796 , 0.02339669, 0.04806345, 0.262158  ],      dtype=float32)
phi_new[0]=array([0.00055184, 0.0339425 , 0.05590081, 0.07988738, 0.0408109 ,
       0.00743632, 0.07810109, 0.01676256, 0.02814333, 0.13296215],
      dtype=float32)


### Использование рассчитанных Фи и n_t в следующем шаге до сходимости

# Доработки

In [34]:
def _create_attention_mask_simple(batch_size, document_bounds, ctx_len):
    """
    Создаёт маску внимания через сравнение позиций.
    Возвращает (batch_size, 2*ctx_len + 1)
    """
    # Позиции в батче
    positions = jnp.arange(batch_size)  # (I,)
    
    # Смещения контекста: [-3, -2, -1, 0, 1, 2, 3]
    context_offsets = jnp.arange(-ctx_len, ctx_len + 1)  # (2C+1,)
    
    # Все позиции контекста для каждой позиции батча
    context_positions = positions[:, None] + context_offsets[None, :]  # (I, 2C+1)
    
    # Определяем, к какому документу принадлежит каждая позиция
    doc_ids = jnp.zeros(batch_size, dtype=jnp.int32)
    for i in range(1, len(document_bounds)):
        doc_ids = doc_ids.at[document_bounds[i-1]:document_bounds[i]].set(i)
    
    # ID документа для каждой позиции контекста (с клиппингом)
    clipped_positions = jnp.clip(context_positions, 0, batch_size - 1)
    context_doc_ids = doc_ids[clipped_positions]  # (I, 2C+1)
    
    # ID документа для центральной позиции
    center_doc_ids = doc_ids[:, None]  # (I, 1)
    
    # Маска: True если контекст в том же документе
    attn_mask = (context_doc_ids == center_doc_ids)  # (I, 2C+1)
    
    return attn_mask

In [43]:
def _create_context_weights(batch_size, document_bounds, ctx_len, gamma=0.6):
    # 1. Геометрические веса контекста
    suffix_weights = np.cumprod(np.full(ctx_len, (1 - gamma))) * gamma
    prefix_weights = suffix_weights[::-1]
    center_weight = np.array([0.0])  # self_aware_context = False
    
    base_weights = np.concatenate([prefix_weights, center_weight, suffix_weights])  # (2C+1,)
    
    # 2. Маска валидности контекста (БЕЗ CLIP!)
    positions = jnp.arange(batch_size)
    context_offsets = jnp.arange(-ctx_len, ctx_len + 1)
    context_positions = positions[:, None] + context_offsets[None, :]
    
    doc_start = jnp.zeros(batch_size, dtype=jnp.int32)
    doc_end = jnp.zeros(batch_size, dtype=jnp.int32)
    
    for i in range(len(document_bounds) - 1):
        start, end = document_bounds[i], document_bounds[i + 1]
        doc_start = doc_start.at[start:end].set(start)
        doc_end = doc_end.at[start:end].set(end)
    
    attn_mask = (
        (context_positions >= doc_start[:, None]) & 
        (context_positions < doc_end[:, None])
    )
    
    # 3. Применяем веса только к валидным позициям + нормировка
    context_matrix = _norm((attn_mask * base_weights).T).T
    
    return context_matrix

# Использование
context_matrix_new = _create_context_weights(
    batch_size=len(tokenized_data),
    document_bounds=document_bounds,
    ctx_len=ctx_len,
    gamma=0.6
)

print(f"{context_matrix_new=}")

context_matrix_new=array([[0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.390625  , 0.        , 0.390625  ,
        0.15625001, 0.06250001],
       [0.        , 0.14285715, 0.35714287, 0.        , 0.35714287,
        0.14285715, 0.        ],
       [0.0625    , 0.15625   , 0.39062497, 0.        , 0.39062497,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.4166667 , 0.        , 0.4166667 ,
        0.16666667, 0.        ],
       [0.        , 0.16666667, 0.4166667 , 0.        , 0.4166667 ,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
    

In [40]:
print(f"{context_matrix_new=}")

context_matrix_new=array([[0.05128205, 0.12820512, 0.3205128 , 0.        , 0.3205128 ,
        0.12820512, 0.05128205],
       [0.05128205, 0.12820512, 0.3205128 , 0.        , 0.3205128 ,
        0.12820512, 0.05128205],
       [0.05405405, 0.13513513, 0.33783782, 0.        , 0.33783782,
        0.13513513, 0.        ],
       [0.0625    , 0.15625   , 0.39062497, 0.        , 0.39062497,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.4166667 , 0.        , 0.4166667 ,
        0.16666667, 0.        ],
       [0.        , 0.16666667, 0.4166667 , 0.        , 0.4166667 ,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
    

In [41]:
print(f"{context_matrix=}")

context_matrix=array([[0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.390625  , 0.        , 0.390625  ,
        0.15625001, 0.06250001],
       [0.        , 0.14285715, 0.35714287, 0.        , 0.35714287,
        0.14285715, 0.        ],
       [0.0625    , 0.15625   , 0.39062497, 0.        , 0.39062497,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.4166667 , 0.        , 0.4166667 ,
        0.16666667, 0.        ],
       [0.        , 0.16666667, 0.4166667 , 0.        , 0.4166667 ,
        0.        , 0.        ],
       [0.1025641 , 0.25641024, 0.6410256 , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        

In [44]:
from sys import path as syspath
from os import path as ospath
import jax.numpy as jnp

from sklearn.datasets import fetch_20newsgroups

import matplotlib.pyplot as plt
import seaborn as sns

syspath.insert(1, r'D:\ARTM\topic-modelling-attention\src')

from cartm.model import ContextTopicModel2
from cartm.preprocessing import DatasetPreprocessor
from time import time

#data = fetch_20newsgroups(data_home='./data/', subset='all').data
with open('./data/test_data.txt') as f:
    data = f.readlines()

preprocessor = DatasetPreprocessor()
tokenized_data, document_bounds = preprocessor.fit_transform(data)

topic_model = ContextTopicModel2(
    vocab_size=len(preprocessor.vocabulary),
    ctx_len=10,
    n_topics=10
)

start_time = time()
topic_model.fit(
    data=tokenized_data,
    ctx_bounds=document_bounds,
    verbose=2,
    seed=42,
)
end_time = time()
print(f"{end_time - start_time}")

ImportError: cannot import name 'ContextTopicModel2' from 'cartm.model' (D:\ARTM\topic-modelling-attention\src\cartm\model.py)

In [47]:
from cartm.model2 import ContextTopicModel2

ModuleNotFoundError: No module named 'cartm.model2'